In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import PhaseGate
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

def qpe_circuit(theta, n_ancilla):
    """
    Quantum Phase Estimation (QPE) using Quantum Fourier Transform (QFT)
    """
    qpe = QuantumCircuit(n_ancilla + 1, n_ancilla)
    
    # Initialize the last qubit to |1>
    qpe.x(n_ancilla)
    
    # Apply Hadamard gates to ancilla qubits
    for qubit in range(n_ancilla):
        qpe.h(qubit)
    
# Apply Phase gates
    for qubit in range(n_ancilla):
        qpe.append()
    
    # Apply inverse QFT
    qpe.append(qft_dagger(n_ancilla), range(n_ancilla))
    
    # Measure ancilla qubits
    qpe.measure(range(n_ancilla), range(n_ancilla))

    qpe.draw('mpl')
    
    return qpe

def qft_dagger(n):
    """ Inverse Quantum Fourier Transform """
    qc = QuantumCircuit(n)
    for qubit in range(n//2):
        qc.swap(qubit, n-qubit-1)
    for j in range(n):
        for k in range(j):
            qc.cp(-np.pi/(2**(j-k)), k, j)
        qc.h(j)
    return qc

aer_sim = AerSimulator()

def run_qpe(theta, n_ancilla, num_shots):
    """Run the QPE algorithm and compute the estimated phase error."""
    qc = qpe_circuit(theta, n_ancilla)
    
    # Transpile and execute
    pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
    qc_transpiled = pm.run(qc)
    
    sampler = Sampler(aer_sim)
    result = sampler.run([qc_transpiled], shots=num_shots).result()
    counts = result[0].data.c.get_counts()
    
    # Get the most probable result
    max_count_key = max(counts, key=counts.get)
    theta_estimate = int(max_count_key, 2) / (2 ** n_ancilla)
    
    return abs(theta - theta_estimate)

# Define test cases
theta_values = [0.5625, 0.1234]
num_shots_list = np.logspace(2, 5, num=10, dtype=int)
n_ancilla_list = [2, 3, 4, 5]

def plot_error_vs_shots():
    """Plot error vs number of shots."""
    plt.figure(figsize=(8,6))
    
    for theta in theta_values:
        errors = []
        for num_shots in num_shots_list:
            error = run_qpe(theta, 4, num_shots)
            errors.append(error)
        plt.plot(num_shots_list, errors, label=f"Theta = {theta}", marker='o')
    
    plt.plot(num_shots_list, 1/num_shots_list, 'k--', label=r'$O(1/N)$')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel("Number of Shots")
    plt.ylabel("Error")
    plt.legend()
    plt.title("Error vs Number of Shots in QPE")
    plt.grid(True)
    plt.show()

def plot_error_vs_ancilla():
    """Plot error vs number of ancilla qubits."""
    plt.figure(figsize=(8,6))
    
    for theta in theta_values:
        errors = []
        for n_ancilla in n_ancilla_list:
            error = run_qpe(theta, n_ancilla, 10000)
            errors.append(error)
        plt.plot(n_ancilla_list, errors, label=f"Theta = {theta}", marker='o')
    
    plt.xlabel("Number of Ancilla Qubits")
    plt.ylabel("Error")
    plt.legend()
    plt.title("Error vs Number of Ancilla Qubits in QPE")
    plt.grid(True)
    plt.show()

# Run the analysis
plot_error_vs_shots()
plot_error_vs_ancilla()


CircuitError: 'The amount of qubit(2)/clbit(0) arguments does not match the gate expectation (1).'

<Figure size 800x600 with 0 Axes>